# Study Domain Map — Mountain Hydrology

Elevation, rivers, cities, state boundaries, the Rio Grande headwaters (S. Colorado)
and northern New Mexico watershed highlights, plus a southwestern-USA locator inset.

Run the cells top to bottom. The first cell only needs to be run once per session
(it caches the Natural Earth vector data used for rivers/cities/borders).

## 1. Cache Natural Earth data (run once)

Requires internet access — run this on a **login node**, not a compute node.
Once cached locally, later cells (and any batch job) can reuse it offline.

In [1]:
import cartopy.io.shapereader as shpreader

_NE_DATASETS = [
    ("10m", "cultural", "populated_places"),
    ("10m", "cultural", "admin_1_states_provinces_lines"),
    ("10m", "cultural", "admin_0_boundary_lines_land"),
    ("10m", "physical", "rivers_lake_centerlines"),
    ("10m", "physical", "lakes"),
    ("50m", "cultural", "admin_1_states_provinces_lines"),
    ("50m", "cultural", "admin_0_boundary_lines_land"),
    ("50m", "physical", "coastline"),
    ("110m", "physical", "land"),
    ("110m", "physical", "ocean"),
]

for res, cat, name in _NE_DATASETS:
    shpreader.natural_earth(resolution=res, category=cat, name=name)

print("Natural Earth data cached.")

/projects/battobrah@xsede.org/software/anaconda/envs/geo_preproc/lib/python3.11/site-packages/cartopy/io/__init__.py:242: DownloadWarning: Downloading: https://naturalearth.s3.amazonaws.com/10m_cultural/ne_10m_populated_places.zip
  warnings.warn(f'Downloading: {url}', DownloadWarning)
/projects/battobrah@xsede.org/software/anaconda/envs/geo_preproc/lib/python3.11/site-packages/cartopy/io/__init__.py:242: DownloadWarning: Downloading: https://naturalearth.s3.amazonaws.com/10m_cultural/ne_10m_admin_1_states_provinces_lines.zip
  warnings.warn(f'Downloading: {url}', DownloadWarning)
/projects/battobrah@xsede.org/software/anaconda/envs/geo_preproc/lib/python3.11/site-packages/cartopy/io/__init__.py:242: DownloadWarning: Downloading: https://naturalearth.s3.amazonaws.com/10m_physical/ne_10m_rivers_lake_centerlines.zip
  warnings.warn(f'Downloading: {url}', DownloadWarning)
/projects/battobrah@xsede.org/software/anaconda/envs/geo_preproc/lib/python3.11/site-packages/cartopy/io/__init__.py:2

Natural Earth data cached.


## 2. Imports

In [2]:
%matplotlib inline

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.pyplot as plt
import numpy as np
import rasterio
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Rectangle
from rasterio.warp import Resampling, calculate_default_transform, reproject

## 3. Domain, highlighted sub-regions, and colormap

In [3]:
# ---- study domain: [min_lon, min_lat, max_lon, max_lat] ----
DOMAIN = dict(min_lon=-114.0, min_lat=30.0, max_lon=-104.0, max_lat=44.0)

# Rio Grande headwaters, southern Colorado (San Juan Mountains / San Luis Valley)
RIO_GRANDE_HEADWATERS = dict(min_lon=-107.7, max_lon=-105.4, min_lat=37.0, max_lat=38.3)

# Northern New Mexico watersheds (Rio Chama / upper Rio Grande)
N_NEW_MEXICO = dict(min_lon=-107.8, max_lon=-105.3, min_lat=35.5, max_lat=37.0)

# Broader southwestern-US extent used for the locator inset
SW_USA_EXTENT = dict(min_lon=-125.0, max_lon=-102.0, min_lat=25.0, max_lat=49.0)

# Only used to label cities — lower this if you want smaller towns to show up
CITY_MIN_POPULATION = 50_000

# Path to your DEM (GeoTIFF/VRT) — EDIT THIS
DEM_PATH = (
    "/scratch/alpine/battobrah@xsede.org/Rice_NMT_Snow/upper_colorado/experiments/"
    "simulations/MSWX_3h_Snow_Project3_3d_upper_colorado_11yr_50hru_full_domain_"
    "downscaled_test/postprocess/dem.vrt"
)

# Where to save the final figure (set to None to skip saving)
OUT_PATH = "study_domain_map.png"

# A hypsometric (green -> yellow -> brown -> white) elevation colormap
HYPSOMETRIC_CMAP = LinearSegmentedColormap.from_list(
    "hypsometric",
    [
        (0.00, "#225E2A"),  # low valleys - dark green
        (0.15, "#559E3C"),
        (0.35, "#AAB55F"),
        (0.55, "#D2AF7D"),
        (0.72, "#96694E"),
        (0.85, "#644434"),
        (1.00, "#FFFFFF"),  # highest peaks - white
    ],
)

## 4. Helper functions

In [4]:
def read_dem_as_geographic(dem_path, bounds):
    """Read a DEM raster, reprojected to plain lat/lon (EPSG:4326) and
    clipped to `bounds`, returning (lon, lat, elevation)."""
    with rasterio.open(dem_path) as src:
        dst_crs = "EPSG:4326"
        transform, width, height = calculate_default_transform(
            src.crs, dst_crs, src.width, src.height, *src.bounds)
        elevation = np.full((height, width), np.nan, dtype=np.float32)
        reproject(
            source=rasterio.band(src, 1),
            destination=elevation,
            src_transform=src.transform, src_crs=src.crs, src_nodata=src.nodata,
            dst_transform=transform, dst_crs=dst_crs, dst_nodata=np.nan,
            resampling=Resampling.average,
        )

    lon = transform.c + np.arange(width) * transform.a
    lat = transform.f + np.arange(height) * transform.e  # descending (north -> south)

    lon_mask = (lon >= bounds["min_lon"]) & (lon <= bounds["max_lon"])
    lat_mask = (lat >= bounds["min_lat"]) & (lat <= bounds["max_lat"])
    lon_clip = lon[lon_mask]
    lat_clip = lat[lat_mask]
    elev_clip = elevation[np.ix_(lat_mask, lon_mask)]
    return lon_clip, lat_clip, elev_clip

In [5]:
def add_dashed_box(ax, bounds, color, label, label_loc="upper"):
    """Draw a dashed rectangle for a highlighted sub-region, with a label."""
    width = bounds["max_lon"] - bounds["min_lon"]
    height = bounds["max_lat"] - bounds["min_lat"]
    rect = Rectangle((bounds["min_lon"], bounds["min_lat"]), width, height,
                      fill=False, edgecolor=color, linewidth=2.2, linestyle="--",
                      transform=ccrs.PlateCarree(), zorder=6)
    ax.add_patch(rect)

    y = bounds["max_lat"] + 0.12 if label_loc == "upper" else bounds["min_lat"] - 0.35
    ax.text(bounds["min_lon"], y, label, transform=ccrs.PlateCarree(),
            fontsize=9, fontweight="bold", color=color, zorder=7,
            bbox=dict(boxstyle="round,pad=0.25", facecolor="white",
                      edgecolor=color, alpha=0.9))

In [6]:
def add_cities(ax, bounds, min_population):
    """Add labeled city markers from Natural Earth's populated-places dataset."""
    import cartopy.io.shapereader as shpreader

    shp_path = shpreader.natural_earth(resolution="10m", category="cultural",
                                        name="populated_places")
    reader = shpreader.Reader(shp_path)
    for record in reader.records():
        lon, lat = record.geometry.x, record.geometry.y
        if not (bounds["min_lon"] <= lon <= bounds["max_lon"] and
                bounds["min_lat"] <= lat <= bounds["max_lat"]):
            continue
        population = record.attributes.get("POP_MAX", 0)
        if population < min_population:
            continue
        name = record.attributes.get("NAME", "")
        ax.plot(lon, lat, marker="o", markersize=4, color="black",
                transform=ccrs.PlateCarree(), zorder=8)
        ax.text(lon + 0.06, lat + 0.06, name, fontsize=7.5, fontweight="bold",
                transform=ccrs.PlateCarree(), zorder=8)

In [7]:
def style_main_axes(ax, bounds, title):
    ax.set_extent([bounds["min_lon"], bounds["max_lon"], bounds["min_lat"], bounds["max_lat"]],
                  crs=ccrs.PlateCarree())
    gridlines = ax.gridlines(draw_labels=True, linewidth=0.4, color="gray",
                              alpha=0.5, linestyle="--")
    gridlines.top_labels = False
    gridlines.right_labels = False
    ax.set_title(title, fontsize=15, fontweight="bold", pad=10)

In [8]:
def build_main_map(ax, lon, lat, elevation, city_min_population):
    # ---- elevation background ----
    mesh = ax.pcolormesh(lon, lat, elevation, cmap=HYPSOMETRIC_CMAP,
                          shading="auto", transform=ccrs.PlateCarree(), zorder=1)
    cbar = plt.colorbar(mesh, ax=ax, orientation="vertical", shrink=0.7, pad=0.02)
    cbar.set_label("Elevation [m]")

    # ---- state boundaries ----
    states = cfeature.NaturalEarthFeature(category="cultural", name="admin_1_states_provinces_lines",
                                           scale="10m", facecolor="none")
    ax.add_feature(states, edgecolor="black", linewidth=0.9, zorder=4)
    ax.add_feature(cfeature.BORDERS.with_scale("10m"), edgecolor="black", linewidth=1.1, zorder=4)

    # ---- rivers ----
    rivers = cfeature.NaturalEarthFeature(category="physical", name="rivers_lake_centerlines",
                                           scale="10m", facecolor="none")
    ax.add_feature(rivers, edgecolor="#1f6fb2", linewidth=0.8, zorder=3)

    lakes = cfeature.NaturalEarthFeature(category="physical", name="lakes", scale="10m",
                                          facecolor="#a6cee3", edgecolor="#1f6fb2")
    ax.add_feature(lakes, linewidth=0.5, zorder=2)

    # ---- cities ----
    add_cities(ax, DOMAIN, city_min_population)

    # ---- highlighted watershed regions ----
    add_dashed_box(ax, RIO_GRANDE_HEADWATERS, color="#e6198c",
                   label="Rio Grande Headwaters (S. Colorado)", label_loc="upper")
    add_dashed_box(ax, N_NEW_MEXICO, color="#7828dc",
                   label="Northern New Mexico Watersheds", label_loc="lower")

    style_main_axes(ax, DOMAIN, "Mountain Hydrology Study Domain")

In [9]:
def build_inset(fig):
    """Small locator map: southwestern USA context with the study domain boxed."""
    inset_ax = fig.add_axes([0.66, 0.66, 0.28, 0.28], projection=ccrs.PlateCarree())
    inset_ax.set_extent([SW_USA_EXTENT["min_lon"], SW_USA_EXTENT["max_lon"],
                         SW_USA_EXTENT["min_lat"], SW_USA_EXTENT["max_lat"]],
                        crs=ccrs.PlateCarree())
    inset_ax.add_feature(cfeature.LAND, facecolor="#f0efe6")
    inset_ax.add_feature(cfeature.OCEAN, facecolor="#cfe6f5")
    inset_ax.add_feature(cfeature.STATES.with_scale("50m"), edgecolor="black", linewidth=0.5)
    inset_ax.add_feature(cfeature.BORDERS.with_scale("50m"), edgecolor="black", linewidth=0.7)
    inset_ax.coastlines(resolution="50m", linewidth=0.5)

    width = DOMAIN["max_lon"] - DOMAIN["min_lon"]
    height = DOMAIN["max_lat"] - DOMAIN["min_lat"]
    rect = Rectangle((DOMAIN["min_lon"], DOMAIN["min_lat"]), width, height,
                      fill=False, edgecolor="red", linewidth=1.8,
                      transform=ccrs.PlateCarree(), zorder=5)
    inset_ax.add_patch(rect)
    inset_ax.set_title("Southwestern USA", fontsize=9)
    for spine in inset_ax.spines.values():
        spine.set_edgecolor("black")
        spine.set_linewidth(1.2)
    return inset_ax

## 5. Read the DEM

This prints the shape and value range so you can immediately see if the DEM
actually covers the full domain, or if part of it will come back empty (NaN).

In [10]:
lon, lat, elevation = read_dem_as_geographic(DEM_PATH, DOMAIN)

print("Elevation grid shape:", elevation.shape)
print("Longitude range:", lon.min(), "to", lon.max())
print("Latitude range: ", lat.min(), "to", lat.max())
print("Elevation range:", np.nanmin(elevation), "to", np.nanmax(elevation), "m")
print("Fraction of domain with valid (non-NaN) elevation:",
      f"{np.mean(np.isfinite(elevation)):.1%}")

Elevation grid shape: (12300, 12000)
Longitude range: -113.9995833333323 to -104.00041666666597
Latitude range:  33.750416666666375 to 43.99958333333269
Elevation range: 166.8 to 4382.7 m
Fraction of domain with valid (non-NaN) elevation: 100.0%


## 6. Build and display the figure

In [11]:
fig = plt.figure(figsize=(12, 11))
main_ax = fig.add_axes([0.06, 0.06, 0.86, 0.86], projection=ccrs.PlateCarree())

build_main_map(main_ax, lon, lat, elevation, CITY_MIN_POPULATION)
build_inset(fig)

plt.show()

: 

: 

## 7. Save to file (optional)

In [ ]:
if OUT_PATH:
    fig.savefig(OUT_PATH, dpi=300, bbox_inches="tight", facecolor="white")
    print(f"Saved: {OUT_PATH}")